# Growth Opportunity Engine

## Product & Growth Intelligence Platform — Notebook 08

This notebook converts validated upstream product, category, and visitor-segment signals into a ranked, decision-oriented **Growth Opportunity Engine**.

### Objective

The engine answers four business questions:

1. **Where are the highest-value growth opportunities?**
2. **How large is the affected audience?**
3. **How severe is the observed conversion friction?**
4. **What action should the business take first?**

### Analytical pipeline

**Upstream intelligence → Opportunity normalization → Opportunity scoring → Classification → Executive prioritization → Validation → CSV export → Business insights**

### Upstream dependencies

This notebook consumes finalized outputs from:

- **Notebook 06 — User Segmentation & Engagement**
- **Notebook 07 — Product & Category Intelligence**

### Main output

`outputs/growth_opportunity_engine_output.csv`

The final output contains the **top 10 actionable opportunities** with their score, priority, business classification, decision, urgency, and recommended action.

> **Analytical note:** Opportunity scores identify and prioritize observed behavioral signals. They do not establish causality or prove that a specific product, category, or segment is the direct cause of a conversion problem.


## 1. Project Paths & Reproducibility

The notebook resolves the project root from the current working directory instead of relying on a machine-specific Windows path.

This keeps the notebook portable when the GitHub repository is cloned to another machine.

In [1]:
from pathlib import Path

# ============================================================
# CELL 1 — PROJECT PATHS
# ============================================================

# Resolve the project root without hardcoding a local Windows path.
# The notebook can therefore be executed from either:
#   1. the project root, or
#   2. the notebooks directory.

CURRENT_DIR = Path.cwd().resolve()

if (CURRENT_DIR / "outputs").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "outputs").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Project root could not be resolved. "
        "Run this notebook from the project root or notebooks directory."
    )

DATA_PATH = PROJECT_ROOT / "data"
OUTPUTS_PATH = PROJECT_ROOT / "outputs"
FIGURES_PATH = PROJECT_ROOT / "figures"

print("PROJECT ROOT:", PROJECT_ROOT)
print("DATA PATH:", DATA_PATH)
print("OUTPUTS PATH:", OUTPUTS_PATH)
print("FIGURES PATH:", FIGURES_PATH)

# Validate the expected project structure before continuing.
assert OUTPUTS_PATH.exists(), "Expected outputs directory was not found."


PROJECT ROOT: D:\Data science portfolio\03_Product_Growth_Intelligence
DATA PATH: D:\Data science portfolio\03_Product_Growth_Intelligence\data
OUTPUTS PATH: D:\Data science portfolio\03_Product_Growth_Intelligence\outputs
FIGURES PATH: D:\Data science portfolio\03_Product_Growth_Intelligence\figures


## 2. Load & Validate Upstream Analytical Inputs

The engine starts by loading finalized analytical outputs and enforcing an **input contract**.

The contract checks that every upstream dataset contains the columns required by the opportunity engine before any scoring or classification is performed.

In [2]:
# ============================================================
# CELL 2 — LOAD AND VALIDATE ANALYTICAL INPUTS
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# Load finalized upstream analytical outputs
# ------------------------------------------------------------

segment_profile = pd.read_csv(
    OUTPUTS_PATH / "06_segment_profile.csv"
)

segment_funnel = pd.read_csv(
    OUTPUTS_PATH / "06_segment_funnel.csv"
)

segment_retention = pd.read_csv(
    OUTPUTS_PATH / "06_segment_retention.csv"
)

segment_opportunity_signals = pd.read_csv(
    OUTPUTS_PATH / "06_segment_opportunity_signals.csv"
)

product_intelligence = pd.read_csv(
    OUTPUTS_PATH / "07_product_intelligence.csv"
)

category_intelligence = pd.read_csv(
    OUTPUTS_PATH / "07_category_intelligence.csv"
)

product_opportunities = pd.read_csv(
    OUTPUTS_PATH / "07_product_opportunities.csv"
)

category_opportunities = pd.read_csv(
    OUTPUTS_PATH / "07_category_opportunities.csv"
)

segment_category_intelligence = pd.read_csv(
    OUTPUTS_PATH / "07_segment_category_intelligence.csv"
)


# ------------------------------------------------------------
# Input contract
# ------------------------------------------------------------

INPUT_CONTRACT = {

    "segment_profile": {
        "frame": segment_profile,
        "required_columns": [
            "segment",
            "visitors",
            "avg_active_days",
            "avg_events",
            "avg_views",
            "avg_carts",
            "avg_transactions",
            "buyer_rate_pct",
            "returning_rate_pct",
        ],
    },

    "segment_funnel": {
        "frame": segment_funnel,
        "required_columns": [
            "segment",
            "visitors",
            "viewed_visitors",
            "cart_visitors",
            "transaction_visitors",
        ],
    },

    "segment_retention": {
        "frame": segment_retention,
        "required_columns": [
            "segment",
        ],
    },

    "segment_opportunity_signals": {
        "frame": segment_opportunity_signals,
        "required_columns": [
            "segment",
        ],
    },

    "product_intelligence": {
        "frame": product_intelligence,
        "required_columns": [
            "itemid",
            "viewed_visitors",
            "cart_visitors",
            "transaction_visitors",
        ],
    },

    "category_intelligence": {
        "frame": category_intelligence,
        "required_columns": [
            "category",
            "viewed_visitors",
            "cart_visitors",
            "transaction_visitors",
        ],
    },

    "product_opportunities": {
        "frame": product_opportunities,
        "required_columns": [
            "itemid",
            "viewed_visitors",
            "view_to_cart_pct",
        ],
    },

    "category_opportunities": {
        "frame": category_opportunities,
        "required_columns": [
            "category",
            "viewed_visitors",
            "view_to_cart_pct",
        ],
    },

    "segment_category_intelligence": {
        "frame": segment_category_intelligence,
        "required_columns": [
            "segment",
            "category",
        ],
    },
}


# ------------------------------------------------------------
# Validate schemas
# ------------------------------------------------------------

validation_results = {}

for name, contract in INPUT_CONTRACT.items():

    frame = contract["frame"]
    required = contract["required_columns"]

    missing_columns = [
        column
        for column in required
        if column not in frame.columns
    ]

    validation_results[name] = len(missing_columns) == 0

    if missing_columns:
        raise ValueError(
            f"Input contract failed for '{name}'. "
            f"Missing columns: {missing_columns}"
        )


# ------------------------------------------------------------
# Input summary
# ------------------------------------------------------------

print("NOTEBOOK 08 — INPUT VALIDATION")
print("=" * 75)

for name, passed in validation_results.items():
    print(
        f"{name:<35} "
        f"{'PASSED' if passed else 'FAILED'}"
    )


print("\nINPUT DATASET SIZES")
print("=" * 75)

for name, contract in INPUT_CONTRACT.items():

    frame = contract["frame"]

    print(
        f"{name:<35} "
        f"{len(frame):>10,} rows × "
        f"{len(frame.columns):>3} columns"
    )


print("\nALL UPSTREAM INPUT CONTRACTS PASSED")

NOTEBOOK 08 — INPUT VALIDATION
segment_profile                     PASSED
segment_funnel                      PASSED
segment_retention                   PASSED
segment_opportunity_signals         PASSED
product_intelligence                PASSED
category_intelligence               PASSED
product_opportunities               PASSED
category_opportunities              PASSED
segment_category_intelligence       PASSED

INPUT DATASET SIZES
segment_profile                              8 rows ×  12 columns
segment_funnel                               8 rows ×   8 columns
segment_retention                            8 rows ×   4 columns
segment_opportunity_signals                  6 rows ×   3 columns
product_intelligence                   235,061 rows ×  16 columns
category_intelligence                    1,087 rows ×  17 columns
product_opportunities                      122 rows ×  16 columns
category_opportunities                      26 rows ×  16 columns
segment_category_intelligence    

## 3. Opportunity Engine Configuration

The configuration documents the analytical thresholds and scoring weights used by the engine.

These settings make the decision framework transparent and reproducible.

In [3]:
# ============================================================
# CELL 3 — OPPORTUNITY ENGINE CONFIGURATION
# ============================================================

# This cell defines transparent, reproducible thresholds used
# by the Growth Opportunity Engine.
#
# The thresholds are intentionally evidence-based and derived
# from the finalized upstream analytical layers.
#
# Important:
# These thresholds identify diagnostic priorities.
# They do NOT establish causality or prove a product defect.


# ------------------------------------------------------------
# Product opportunity configuration
# ------------------------------------------------------------

PRODUCT_MIN_VIEWERS = 100
PRODUCT_INTEREST_QUANTILE = 0.90


# ------------------------------------------------------------
# Category opportunity configuration
# ------------------------------------------------------------

CATEGORY_MIN_VIEWERS = 500
CATEGORY_INTEREST_QUANTILE = 0.90


# ------------------------------------------------------------
# Segment opportunity configuration
# ------------------------------------------------------------

# A segment is considered materially sized when it contains
# at least 5% of the observed visitor population.
SEGMENT_MIN_POPULATION_SHARE = 0.05


# ------------------------------------------------------------
# Evidence thresholds
# ------------------------------------------------------------

# These are aligned with the finalized Notebook 06 and
# Notebook 07 opportunity definitions.

LOW_TRANSACTION_PARTICIPATION_PCT = 1.0
LOW_VIEW_TO_TRANSACTION_PCT = 1.0
LOW_D7_RETENTION_PCT = 1.0


# ------------------------------------------------------------
# Opportunity scoring weights
# ------------------------------------------------------------

# The engine combines three dimensions:
#
# 1. Reach       → how much observed population is affected
# 2. Friction    → evidence of weak progression
# 3. Confidence  → strength of the observed signal
#
# Weights sum to 1.00.

REACH_WEIGHT = 0.40
FRICTION_WEIGHT = 0.40
CONFIDENCE_WEIGHT = 0.20


# ------------------------------------------------------------
# Validation of configuration
# ------------------------------------------------------------

assert PRODUCT_MIN_VIEWERS > 0
assert CATEGORY_MIN_VIEWERS > 0

assert 0 < PRODUCT_INTEREST_QUANTILE < 1
assert 0 < CATEGORY_INTEREST_QUANTILE < 1

assert 0 < SEGMENT_MIN_POPULATION_SHARE <= 1

assert LOW_TRANSACTION_PARTICIPATION_PCT >= 0
assert LOW_VIEW_TO_TRANSACTION_PCT >= 0
assert LOW_D7_RETENTION_PCT >= 0

assert REACH_WEIGHT >= 0
assert FRICTION_WEIGHT >= 0
assert CONFIDENCE_WEIGHT >= 0

assert (
    abs(
        REACH_WEIGHT
        + FRICTION_WEIGHT
        + CONFIDENCE_WEIGHT
        - 1.0
    )
    < 1e-9
)


# ------------------------------------------------------------
# Configuration summary
# ------------------------------------------------------------

print("GROWTH OPPORTUNITY ENGINE — CONFIGURATION")
print("=" * 70)

print("\nPRODUCT OPPORTUNITY")
print(f"Minimum viewed visitors : {PRODUCT_MIN_VIEWERS:,}")
print(f"Interest percentile     : {PRODUCT_INTEREST_QUANTILE:.0%}")

print("\nCATEGORY OPPORTUNITY")
print(f"Minimum viewed visitors : {CATEGORY_MIN_VIEWERS:,}")
print(f"Interest percentile     : {CATEGORY_INTEREST_QUANTILE:.0%}")

print("\nSEGMENT OPPORTUNITY")
print(
    "Material population     : "
    f"{SEGMENT_MIN_POPULATION_SHARE:.0%} of observed visitors"
)

print("\nEVIDENCE THRESHOLDS")
print(
    "Low transaction participation : "
    f"< {LOW_TRANSACTION_PARTICIPATION_PCT:.1f}%"
)
print(
    "Low view-to-transaction       : "
    f"< {LOW_VIEW_TO_TRANSACTION_PCT:.1f}%"
)
print(
    "Low D7 retention              : "
    f"< {LOW_D7_RETENTION_PCT:.1f}%"
)

print("\nSCORING WEIGHTS")
print(f"Reach       : {REACH_WEIGHT:.0%}")
print(f"Friction    : {FRICTION_WEIGHT:.0%}")
print(f"Confidence  : {CONFIDENCE_WEIGHT:.0%}")

print("\nCONFIGURATION VALIDATION: PASSED")

GROWTH OPPORTUNITY ENGINE — CONFIGURATION

PRODUCT OPPORTUNITY
Minimum viewed visitors : 100
Interest percentile     : 90%

CATEGORY OPPORTUNITY
Minimum viewed visitors : 500
Interest percentile     : 90%

SEGMENT OPPORTUNITY
Material population     : 5% of observed visitors

EVIDENCE THRESHOLDS
Low transaction participation : < 1.0%
Low view-to-transaction       : < 1.0%
Low D7 retention              : < 1.0%

SCORING WEIGHTS
Reach       : 40%
Friction    : 40%
Confidence  : 20%

CONFIGURATION VALIDATION: PASSED


## 4. Normalize Opportunity Signals

Segment, product, and category opportunities have different upstream structures.

This stage converts them into a common schema so that the same downstream scoring framework can evaluate all opportunity types consistently.

In [4]:
# ============================================================
# CELL 4 — NORMALIZE UPSTREAM OPPORTUNITY SIGNALS
# ============================================================

# Standardize opportunity evidence from the upstream analytical
# layers so the Growth Opportunity Engine can score each
# opportunity consistently.
#
# Upstream sources:
#   Notebook 06 → visitor segment opportunity signals
#   Notebook 07 → product opportunity signals
#   Notebook 07 → category opportunity signals


# ------------------------------------------------------------
# 1. Segment opportunity signals
# ------------------------------------------------------------

segment_signals = segment_opportunity_signals.copy()

segment_signals = segment_signals.rename(
    columns={
        "visitors": "affected_visitors",
        "signals": "evidence"
    }
)

segment_signals["opportunity_type"] = "segment"

segment_signals["entity_id"] = (
    segment_signals["segment"].astype(str)
)

segment_signals = segment_signals[
    [
        "opportunity_type",
        "entity_id",
        "segment",
        "affected_visitors",
        "evidence"
    ]
].copy()


# ------------------------------------------------------------
# 2. Product opportunity signals
# ------------------------------------------------------------

product_signals = product_opportunities.copy()

product_signals["opportunity_type"] = "product"

product_signals["entity_id"] = (
    product_signals["itemid"].astype(str)
)

product_signals["affected_visitors"] = (
    product_signals["viewed_visitors"]
)

product_signals["evidence"] = (
    product_signals["opportunity_signal"]
)

product_signals = product_signals[
    [
        "opportunity_type",
        "entity_id",
        "itemid",
        "categoryid",
        "affected_visitors",
        "viewed_visitors",
        "cart_visitors",
        "transaction_visitors",
        "view_to_cart_pct",
        "cart_to_transaction_pct",
        "evidence"
    ]
].copy()


# ------------------------------------------------------------
# 3. Category opportunity signals
# ------------------------------------------------------------

category_signals = category_opportunities.copy()

category_signals["opportunity_type"] = "category"

# Notebook 07 defines category opportunities from category_base,
# where the business-facing category identifier is the `category`
# field. Therefore the category label is used as the stable
# opportunity entity identifier here.

category_signals["entity_id"] = (
    category_signals["category"].astype(str)
)

category_signals["affected_visitors"] = (
    category_signals["viewed_visitors"]
)

category_signals["evidence"] = (
    category_signals["opportunity_signal"]
)

category_signals = category_signals[
    [
        "opportunity_type",
        "entity_id",
        "category",
        "affected_visitors",
        "viewed_visitors",
        "cart_visitors",
        "transaction_visitors",
        "view_to_cart_pct",
        "cart_to_transaction_pct",
        "evidence"
    ]
].copy()


# ------------------------------------------------------------
# 4. Structural validation
# ------------------------------------------------------------

required_inputs = {
    "segment_opportunity_signals": segment_opportunity_signals,
    "product_opportunities": product_opportunities,
    "category_opportunities": category_opportunities
}

for name, frame in required_inputs.items():

    if frame is None or frame.empty:
        raise ValueError(
            f"Required upstream input '{name}' is empty."
        )


# Ensure standardized tables contain no duplicate columns.

assert not segment_signals.columns.duplicated().any()
assert not product_signals.columns.duplicated().any()
assert not category_signals.columns.duplicated().any()


# Ensure affected visitor populations are non-negative.

assert (
    segment_signals["affected_visitors"]
    .fillna(0)
    .ge(0)
    .all()
)

assert (
    product_signals["affected_visitors"]
    .fillna(0)
    .ge(0)
    .all()
)

assert (
    category_signals["affected_visitors"]
    .fillna(0)
    .ge(0)
    .all()
)


# Ensure every opportunity has an entity identifier.

assert (
    segment_signals["entity_id"]
    .notna()
    .all()
)

assert (
    product_signals["entity_id"]
    .notna()
    .all()
)

assert (
    category_signals["entity_id"]
    .notna()
    .all()
)


# ------------------------------------------------------------
# 5. Normalization summary
# ------------------------------------------------------------

total_signals = (
    len(segment_signals)
    + len(product_signals)
    + len(category_signals)
)

print("UPSTREAM OPPORTUNITY SIGNALS NORMALIZED")
print("=" * 70)

print(
    f"Segment signals       : {len(segment_signals):,}"
)

print(
    f"Product signals       : {len(product_signals):,}"
)

print(
    f"Category signals      : {len(category_signals):,}"
)

print(
    f"Total opportunity signals : {total_signals:,}"
)

print("\nNORMALIZATION VALIDATION: PASSED")

UPSTREAM OPPORTUNITY SIGNALS NORMALIZED
Segment signals       : 6
Product signals       : 122
Category signals      : 26
Total opportunity signals : 154

NORMALIZATION VALIDATION: PASSED


## 5. Build the Unified Opportunity Table

The normalized opportunity signals are combined into one analytical table.

This becomes the central dataset for scoring, prioritization, validation, and executive reporting.

In [5]:
# ============================================================
# CELL 5 — BUILD UNIFIED OPPORTUNITY TABLE
# ============================================================

# Combine segment, product, and category opportunity signals
# into one standardized analytical table.
#
# The table preserves the evidence required by the downstream
# scoring engine while keeping a consistent structure across
# all opportunity types.


# ------------------------------------------------------------
# 1. Standardize segment opportunities
# ------------------------------------------------------------

segment_unified = segment_signals[
    [
        "opportunity_type",
        "entity_id",
        "affected_visitors",
        "evidence"
    ]
].copy()

segment_unified["entity_label"] = (
    segment_signals["segment"].astype(str)
)

segment_unified["viewed_visitors"] = np.nan
segment_unified["cart_visitors"] = np.nan
segment_unified["transaction_visitors"] = np.nan
segment_unified["view_to_cart_pct"] = np.nan
segment_unified["cart_to_transaction_pct"] = np.nan


# ------------------------------------------------------------
# 2. Standardize product opportunities
# ------------------------------------------------------------

product_unified = product_signals[
    [
        "opportunity_type",
        "entity_id",
        "affected_visitors",
        "viewed_visitors",
        "cart_visitors",
        "transaction_visitors",
        "view_to_cart_pct",
        "cart_to_transaction_pct",
        "evidence"
    ]
].copy()

product_unified["entity_label"] = (
    product_signals["itemid"].astype(str)
)


# ------------------------------------------------------------
# 3. Standardize category opportunities
# ------------------------------------------------------------

category_unified = category_signals[
    [
        "opportunity_type",
        "entity_id",
        "affected_visitors",
        "viewed_visitors",
        "cart_visitors",
        "transaction_visitors",
        "view_to_cart_pct",
        "cart_to_transaction_pct",
        "evidence"
    ]
].copy()

category_unified["entity_label"] = (
    category_signals["category"].astype(str)
)


# ------------------------------------------------------------
# 4. Combine all opportunity types
# ------------------------------------------------------------

opportunity_signals = pd.concat(
    [
        segment_unified,
        product_unified,
        category_unified
    ],
    ignore_index=True,
    sort=False
)


# ------------------------------------------------------------
# 5. Enforce final column order
# ------------------------------------------------------------

opportunity_signals = opportunity_signals[
    [
        "opportunity_type",
        "entity_id",
        "entity_label",
        "affected_visitors",
        "viewed_visitors",
        "cart_visitors",
        "transaction_visitors",
        "view_to_cart_pct",
        "cart_to_transaction_pct",
        "evidence"
    ]
].copy()


# ------------------------------------------------------------
# 6. Data-quality validation
# ------------------------------------------------------------

assert len(opportunity_signals) == total_signals

assert (
    opportunity_signals["opportunity_type"]
    .isin(["segment", "product", "category"])
    .all()
)

assert (
    opportunity_signals["entity_id"]
    .notna()
    .all()
)

assert (
    opportunity_signals["entity_label"]
    .notna()
    .all()
)

assert (
    opportunity_signals["affected_visitors"]
    .fillna(0)
    .ge(0)
    .all()
)


# Validate percentage measures wherever they exist.

for column in [
    "view_to_cart_pct",
    "cart_to_transaction_pct"
]:
    valid_values = opportunity_signals[column].dropna()

    assert (
        valid_values.between(0, 100).all()
    )


# ------------------------------------------------------------
# 7. Validate opportunity counts by type
# ------------------------------------------------------------

expected_counts = {
    "segment": len(segment_signals),
    "product": len(product_signals),
    "category": len(category_signals)
}

actual_counts = (
    opportunity_signals["opportunity_type"]
    .value_counts()
    .to_dict()
)

assert actual_counts == expected_counts


# ------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------

print("UNIFIED OPPORTUNITY TABLE CREATED")
print("=" * 70)

print(
    f"Total opportunities : {len(opportunity_signals):,}"
)

print(
    f"Segment opportunities : "
    f"{actual_counts.get('segment', 0):,}"
)

print(
    f"Product opportunities : "
    f"{actual_counts.get('product', 0):,}"
)

print(
    f"Category opportunities : "
    f"{actual_counts.get('category', 0):,}"
)

print(
    f"Columns : {len(opportunity_signals.columns)}"
)

print("\nOPPORTUNITY TABLE VALIDATION: PASSED")

display(
    opportunity_signals.head(10)
)

UNIFIED OPPORTUNITY TABLE CREATED
Total opportunities : 154
Segment opportunities : 6
Product opportunities : 122
Category opportunities : 26
Columns : 10

OPPORTUNITY TABLE VALIDATION: PASSED


,opportunity_type,entity_id,entity_label,affected_visitors,viewed_visitors,cart_visitors,transaction_visitors,view_to_cart_pct,cart_to_transaction_pct,evidence
0,segment,High-Intent Visitors,High-Intent Visitors,5823.0,NaN,NaN,NaN,NaN,NaN,higher cart intent without transaction; low tr...
1,segment,Cart Abandoners,Cart Abandoners,21323.0,NaN,NaN,NaN,NaN,NaN,cart activity without transaction; low transac...
2,segment,At-Risk Visitors,At-Risk Visitors,113283.0,NaN,NaN,NaN,NaN,NaN,material visitor population; recent inactivity...
3,segment,New Visitors,New Visitors,49219.0,NaN,NaN,NaN,NaN,NaN,low transaction participation
4,segment,Highly Engaged Browsers,Highly Engaged Browsers,5057.0,NaN,NaN,NaN,NaN,NaN,repeated browsing without cart/transaction; lo...
5,segment,Other Visitors,Other Visitors,1201156.0,NaN,NaN,NaN,NaN,NaN,material visitor population; low transaction p...
6,product,187946,187946,2911.0,2911.0,2.0,0.0,0.068705,0.0,High observed product interest with below-medi...
7,product,5411,5411,2078.0,2078.0,9.0,0.0,0.433109,0.0,High observed product interest with below-medi...
8,product,370653,370653,1577.0,1577.0,0.0,0.0,0.000000,NaN,High observed product interest with below-medi...
9,product,96924,96924,1359.0,1359.0,0.0,0.0,0.000000,NaN,High observed product interest with below-medi...


## 6. Opportunity Scoring Engine

Each opportunity receives three component scores:

- **Reach** — size of the affected audience
- **Friction** — strength of the observed conversion problem
- **Confidence** — completeness of the available evidence

The final opportunity score combines these dimensions using the configured weights.

In [6]:
# ============================================================
# CELL 6 — OPPORTUNITY SCORING ENGINE
# ============================================================

# The scoring engine ranks opportunities using three dimensions:
#
#   Reach      → size of the affected audience
#   Friction   → strength of the observed conversion problem
#   Confidence → reliability/completeness of the evidence
#
# Final score:
#
#   Opportunity Score =
#       40% Reach
#     + 40% Friction
#     + 20% Confidence
#
# All component scores are normalized to 0–100.


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

SCORING_WEIGHTS = {
    "reach": 0.40,
    "friction": 0.40,
    "confidence": 0.20
}

assert abs(sum(SCORING_WEIGHTS.values()) - 1.0) < 1e-9


# ------------------------------------------------------------
# 2. Create scoring dataset
# ------------------------------------------------------------

scored_opportunities = opportunity_signals.copy()


# ------------------------------------------------------------
# 3. REACH SCORE
# ------------------------------------------------------------
# Log transformation prevents extremely large populations
# from dominating the ranking.

reach_population = (
    scored_opportunities["affected_visitors"]
    .fillna(0)
    .clip(lower=0)
)

reach_log = np.log1p(reach_population)

reach_min = reach_log.min()
reach_max = reach_log.max()

if reach_max > reach_min:
    scored_opportunities["reach_score"] = (
        (reach_log - reach_min)
        / (reach_max - reach_min)
        * 100
    )
else:
    scored_opportunities["reach_score"] = 100.0


# ------------------------------------------------------------
# 4. FRICTION SCORE
# ------------------------------------------------------------
# For product/category opportunities, observed funnel
# conversion rates provide direct evidence of friction.
#
# Lower conversion → higher friction.
#
# Segment opportunities do not contain product funnel rates,
# therefore their friction is derived from the opportunity
# evidence itself.

view_to_cart = (
    scored_opportunities["view_to_cart_pct"]
    .fillna(0)
    .clip(lower=0, upper=100)
)

cart_to_transaction = (
    scored_opportunities["cart_to_transaction_pct"]
    .fillna(0)
    .clip(lower=0, upper=100)
)

# For product/category opportunities, use the weaker funnel
# transition as the primary friction signal.

funnel_friction = (
    100
    - np.minimum(
        view_to_cart,
        cart_to_transaction
    )
)

# Segment opportunities already represent identified
# behavioral problems. Give them a strong friction signal.

segment_mask = (
    scored_opportunities["opportunity_type"]
    == "segment"
)

scored_opportunities["friction_score"] = funnel_friction

scored_opportunities.loc[
    segment_mask,
    "friction_score"
] = 100.0


# ------------------------------------------------------------
# 5. CONFIDENCE SCORE
# ------------------------------------------------------------
# Confidence reflects how much structured evidence is
# available for the opportunity.
#
# Product/category opportunities with funnel metrics receive
# stronger evidence confidence.
#
# Segment opportunities rely on population + qualitative
# opportunity evidence.

has_population = (
    scored_opportunities["affected_visitors"]
    .notna()
    & scored_opportunities["affected_visitors"].gt(0)
)

has_funnel_evidence = (
    scored_opportunities["view_to_cart_pct"].notna()
    | scored_opportunities["cart_to_transaction_pct"].notna()
)

has_evidence_text = (
    scored_opportunities["evidence"]
    .notna()
    & scored_opportunities["evidence"]
    .astype(str)
    .str.strip()
    .ne("")
)

scored_opportunities["confidence_score"] = (
    has_population.astype(float) * 40
    + has_funnel_evidence.astype(float) * 40
    + has_evidence_text.astype(float) * 20
)

# Segment opportunities have structured population and
# evidence text, so their maximum evidence confidence is 60
# under the generic rule above. Normalize their available
# evidence to maintain comparability.

scored_opportunities.loc[
    segment_mask,
    "confidence_score"
] = np.where(
    has_population[segment_mask]
    & has_evidence_text[segment_mask],
    100.0,
    60.0
)


# ------------------------------------------------------------
# 6. FINAL OPPORTUNITY SCORE
# ------------------------------------------------------------

scored_opportunities["opportunity_score"] = (
    scored_opportunities["reach_score"]
    * SCORING_WEIGHTS["reach"]
    +
    scored_opportunities["friction_score"]
    * SCORING_WEIGHTS["friction"]
    +
    scored_opportunities["confidence_score"]
    * SCORING_WEIGHTS["confidence"]
)


# ------------------------------------------------------------
# 7. OPPORTUNITY PRIORITY
# ------------------------------------------------------------

scored_opportunities["priority"] = pd.cut(
    scored_opportunities["opportunity_score"],
    bins=[-np.inf, 40, 60, 80, np.inf],
    labels=[
        "Low",
        "Medium",
        "High",
        "Critical"
    ]
)


# ------------------------------------------------------------
# 8. Rank opportunities
# ------------------------------------------------------------

scored_opportunities = (
    scored_opportunities
    .sort_values(
        [
            "opportunity_score",
            "affected_visitors"
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

scored_opportunities["opportunity_rank"] = (
    np.arange(1, len(scored_opportunities) + 1)
)


# ------------------------------------------------------------
# 9. Validation
# ------------------------------------------------------------

assert len(scored_opportunities) == 154

assert (
    scored_opportunities["reach_score"]
    .between(0, 100)
    .all()
)

assert (
    scored_opportunities["friction_score"]
    .between(0, 100)
    .all()
)

assert (
    scored_opportunities["confidence_score"]
    .between(0, 100)
    .all()
)

assert (
    scored_opportunities["opportunity_score"]
    .between(0, 100)
    .all()
)

assert (
    scored_opportunities["opportunity_rank"]
    .is_unique
)

assert (
    scored_opportunities["opportunity_rank"]
    .min()
    == 1
)

assert (
    scored_opportunities["opportunity_rank"]
    .max()
    == 154
)


# ------------------------------------------------------------
# 10. Final scoring summary
# ------------------------------------------------------------

print("OPPORTUNITY SCORING ENGINE")
print("=" * 70)

print(
    f"Opportunities scored : "
    f"{len(scored_opportunities):,}"
)

print(
    f"Reach weight         : "
    f"{SCORING_WEIGHTS['reach']:.0%}"
)

print(
    f"Friction weight      : "
    f"{SCORING_WEIGHTS['friction']:.0%}"
)

print(
    f"Confidence weight    : "
    f"{SCORING_WEIGHTS['confidence']:.0%}"
)

print("\nPRIORITY DISTRIBUTION")
print("-" * 70)

print(
    scored_opportunities["priority"]
    .value_counts()
    .sort_index()
)

print("\nSCORING VALIDATION: PASSED")

print("\nTOP 10 OPPORTUNITIES")
print("-" * 70)

display(
    scored_opportunities[
        [
            "opportunity_rank",
            "opportunity_type",
            "entity_label",
            "affected_visitors",
            "reach_score",
            "friction_score",
            "confidence_score",
            "opportunity_score",
            "priority"
        ]
    ].head(10)
)

OPPORTUNITY SCORING ENGINE
Opportunities scored : 154
Reach weight         : 40%
Friction weight      : 40%
Confidence weight    : 20%

PRIORITY DISTRIBUTION
----------------------------------------------------------------------
priority
Low           0
Medium       12
High        135
Critical      7
Name: count, dtype: int64

SCORING VALIDATION: PASSED

TOP 10 OPPORTUNITIES
----------------------------------------------------------------------


,opportunity_rank,opportunity_type,entity_label,affected_visitors,reach_score,friction_score,confidence_score,opportunity_score,priority
0,1,segment,Other Visitors,1201156.0,100.000000,100.000000,100.0,100.000000,Critical
1,2,category,Uncategorized,197747.0,78.233992,99.745129,100.0,91.191648,Critical
2,3,segment,At-Risk Visitors,113283.0,71.512578,100.000000,100.0,88.605031,Critical
3,4,segment,New Visitors,49219.0,61.455136,100.000000,100.0,84.582054,Critical
4,5,category,1483,40486.0,59.098591,97.127402,100.0,82.490397,Critical
5,6,category,491,37763.0,58.258563,96.883193,100.0,82.056702,Critical
6,7,segment,Cart Abandoners,21323.0,51.363080,100.000000,100.0,80.545232,Critical
7,8,category,48,17016.0,48.640947,97.220263,100.0,78.344484,High
8,9,category,5,15036.0,47.148506,97.073690,100.0,77.688878,High
9,10,category,1578,13650.0,45.981799,97.860806,100.0,77.537042,High


## 7. Opportunity Classification & Recommended Actions

The numerical score is translated into business-readable classifications, decisions, urgency levels, and recommended actions.

This is the layer that turns analytical signals into **decision support**.

In [7]:
# ============================================================
# CELL 7 — OPPORTUNITY CLASSIFICATION & RECOMMENDED ACTION
# ============================================================

# This layer converts the numerical opportunity score into
# business-readable opportunity classes and recommended actions.
#
# The purpose is to make the engine decision-oriented rather
# than simply producing a ranked table.


# ------------------------------------------------------------
# 1. Work from the scored opportunity table
# ------------------------------------------------------------

opportunity_actions = scored_opportunities.copy()


# ------------------------------------------------------------
# 2. Define opportunity classification logic
# ------------------------------------------------------------

def classify_opportunity(row):

    opportunity_type = row["opportunity_type"]
    friction = row["friction_score"]
    reach = row["reach_score"]
    confidence = row["confidence_score"]

    # Segment opportunities
    if opportunity_type == "segment":

        if friction >= 80 and reach >= 70:
            return "High-Reach Segment Friction"

        if friction >= 80:
            return "Segment Conversion Risk"

        return "Segment Engagement Opportunity"

    # Product opportunities
    if opportunity_type == "product":

        if friction >= 80 and reach >= 70:
            return "High-Reach Product Friction"

        if friction >= 80:
            return "Product Conversion Friction"

        if reach >= 70:
            return "High-Reach Product Opportunity"

        return "Product Optimization Opportunity"

    # Category opportunities
    if opportunity_type == "category":

        if friction >= 80 and reach >= 70:
            return "High-Reach Category Friction"

        if friction >= 80:
            return "Category Conversion Friction"

        if reach >= 70:
            return "High-Reach Category Opportunity"

        return "Category Optimization Opportunity"

    return "General Growth Opportunity"


opportunity_actions["opportunity_class"] = (
    opportunity_actions.apply(
        classify_opportunity,
        axis=1
    )
)


# ------------------------------------------------------------
# 3. Recommended action logic
# ------------------------------------------------------------

def recommend_action(row):

    opportunity_type = row["opportunity_type"]
    friction = row["friction_score"]
    reach = row["reach_score"]

    # Segment actions
    if opportunity_type == "segment":

        if friction >= 80 and reach >= 70:
            return (
                "Prioritize targeted intervention for this "
                "high-reach segment and investigate the "
                "dominant conversion barrier."
            )

        if friction >= 80:
            return (
                "Investigate segment-level conversion friction "
                "and test targeted engagement interventions."
            )

        return (
            "Test engagement and personalization strategies "
            "for this visitor segment."
        )

    # Product actions
    if opportunity_type == "product":

        if friction >= 80 and reach >= 70:
            return (
                "Prioritize product-level conversion optimization "
                "and investigate the largest funnel drop-off."
            )

        if friction >= 80:
            return (
                "Investigate product funnel friction and test "
                "changes to improve cart or transaction participation."
            )

        if reach >= 70:
            return (
                "Prioritize this product because of its large "
                "observed audience and evaluate conversion upside."
            )

        return (
            "Monitor product performance and test targeted "
            "conversion improvements."
        )

    # Category actions
    if opportunity_type == "category":

        if friction >= 80 and reach >= 70:
            return (
                "Prioritize category-level conversion optimization "
                "and investigate the dominant funnel barrier."
            )

        if friction >= 80:
            return (
                "Investigate category conversion friction and "
                "test improvements across the category journey."
            )

        if reach >= 70:
            return (
                "Prioritize this category because of its broad "
                "audience reach and evaluate conversion upside."
            )

        return (
            "Monitor category performance and test targeted "
            "conversion improvements."
        )

    return (
        "Investigate the opportunity and validate the underlying "
        "behavioral signal before intervention."
    )


opportunity_actions["recommended_action"] = (
    opportunity_actions.apply(
        recommend_action,
        axis=1
    )
)


# ------------------------------------------------------------
# 4. Decision urgency
# ------------------------------------------------------------

def assign_urgency(row):

    score = row["opportunity_score"]

    if score >= 80:
        return "Immediate"

    if score >= 60:
        return "Prioritize"

    if score >= 40:
        return "Monitor"

    return "Low Priority"


opportunity_actions["decision_urgency"] = (
    opportunity_actions.apply(
        assign_urgency,
        axis=1
    )
)


# ------------------------------------------------------------
# 5. Business decision label
# ------------------------------------------------------------

opportunity_actions["decision"] = np.select(
    [
        opportunity_actions["opportunity_score"] >= 80,
        opportunity_actions["opportunity_score"] >= 60,
        opportunity_actions["opportunity_score"] >= 40
    ],
    [
        "ACT NOW",
        "PRIORITIZE",
        "MONITOR"
    ],
    default="LOW PRIORITY"
)


# ------------------------------------------------------------
# 6. Validation
# ------------------------------------------------------------

assert len(opportunity_actions) == 154

assert (
    opportunity_actions["opportunity_class"]
    .notna()
    .all()
)

assert (
    opportunity_actions["recommended_action"]
    .notna()
    .all()
)

assert (
    opportunity_actions["decision_urgency"]
    .notna()
    .all()
)

assert (
    opportunity_actions["decision"]
    .notna()
    .all()
)

assert set(
    opportunity_actions["decision"]
).issubset(
    {
        "ACT NOW",
        "PRIORITIZE",
        "MONITOR",
        "LOW PRIORITY"
    }
)


# ------------------------------------------------------------
# 7. Final decision summary
# ------------------------------------------------------------

print("OPPORTUNITY CLASSIFICATION ENGINE")
print("=" * 70)

print(
    f"Opportunities classified : "
    f"{len(opportunity_actions):,}"
)

print("\nDECISION DISTRIBUTION")
print("-" * 70)

print(
    opportunity_actions["decision"]
    .value_counts()
)


print("\nOPPORTUNITY CLASS DISTRIBUTION")
print("-" * 70)

print(
    opportunity_actions["opportunity_class"]
    .value_counts()
)


print("\nCLASSIFICATION VALIDATION: PASSED")


# ------------------------------------------------------------
# 8. Top business opportunities
# ------------------------------------------------------------

print("\nTOP 10 BUSINESS OPPORTUNITIES")
print("-" * 70)

display(
    opportunity_actions[
        [
            "opportunity_rank",
            "opportunity_type",
            "entity_label",
            "affected_visitors",
            "opportunity_score",
            "priority",
            "opportunity_class",
            "decision",
            "decision_urgency",
            "recommended_action"
        ]
    ].head(10)
)

OPPORTUNITY CLASSIFICATION ENGINE
Opportunities classified : 154

DECISION DISTRIBUTION
----------------------------------------------------------------------
decision
PRIORITIZE    136
MONITOR        11
ACT NOW         7
Name: count, dtype: int64

OPPORTUNITY CLASS DISTRIBUTION
----------------------------------------------------------------------
opportunity_class
Product Conversion Friction     122
Category Conversion Friction     25
Segment Conversion Risk           4
High-Reach Segment Friction       2
High-Reach Category Friction      1
Name: count, dtype: int64

CLASSIFICATION VALIDATION: PASSED

TOP 10 BUSINESS OPPORTUNITIES
----------------------------------------------------------------------


,opportunity_rank,opportunity_type,entity_label,affected_visitors,opportunity_score,priority,opportunity_class,decision,decision_urgency,recommended_action
0,1,segment,Other Visitors,1201156.0,100.000000,Critical,High-Reach Segment Friction,ACT NOW,Immediate,Prioritize targeted intervention for this high...
1,2,category,Uncategorized,197747.0,91.191648,Critical,High-Reach Category Friction,ACT NOW,Immediate,Prioritize category-level conversion optimizat...
2,3,segment,At-Risk Visitors,113283.0,88.605031,Critical,High-Reach Segment Friction,ACT NOW,Immediate,Prioritize targeted intervention for this high...
3,4,segment,New Visitors,49219.0,84.582054,Critical,Segment Conversion Risk,ACT NOW,Immediate,Investigate segment-level conversion friction ...
4,5,category,1483,40486.0,82.490397,Critical,Category Conversion Friction,ACT NOW,Immediate,Investigate category conversion friction and t...
5,6,category,491,37763.0,82.056702,Critical,Category Conversion Friction,ACT NOW,Immediate,Investigate category conversion friction and t...
6,7,segment,Cart Abandoners,21323.0,80.545232,Critical,Segment Conversion Risk,ACT NOW,Immediate,Investigate segment-level conversion friction ...
7,8,category,48,17016.0,78.344484,High,Category Conversion Friction,PRIORITIZE,Prioritize,Investigate category conversion friction and t...
8,9,category,5,15036.0,77.688878,High,Category Conversion Friction,PRIORITIZE,Prioritize,Investigate category conversion friction and t...
9,10,category,1578,13650.0,77.537042,High,Category Conversion Friction,PRIORITIZE,Prioritize,Investigate category conversion friction and t...


## 8. Executive Opportunity View

The full opportunity table is ranked and converted into an executive-facing view.

The top opportunities are surfaced with the information required for business review and downstream dashboarding.

In [8]:
# ============================================================
# CELL 8 — EXECUTIVE OPPORTUNITY VIEW
# ============================================================

# Create a compact executive view of the highest-value
# opportunities for business decision-making.
#
# This table is designed for downstream dashboarding,
# reporting, and portfolio presentation.


# ------------------------------------------------------------
# 1. Create executive opportunity table
# ------------------------------------------------------------

executive_opportunities = (
    opportunity_actions
    .sort_values(
        by=[
            "opportunity_score",
            "affected_visitors"
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
    .copy()
)


# ------------------------------------------------------------
# 2. Rebuild opportunity rank after final sorting
# ------------------------------------------------------------

executive_opportunities["opportunity_rank"] = (
    executive_opportunities.index + 1
)


# ------------------------------------------------------------
# 3. Select business-facing columns
# ------------------------------------------------------------

executive_columns = [
    "opportunity_rank",
    "opportunity_type",
    "entity_id",
    "entity_label",
    "affected_visitors",
    "opportunity_score",
    "priority",
    "opportunity_class",
    "decision",
    "decision_urgency",
    "recommended_action"
]


executive_opportunities = (
    executive_opportunities[
        executive_columns
    ]
)


# ------------------------------------------------------------
# 4. Executive Top 10
# ------------------------------------------------------------

executive_top_10 = (
    executive_opportunities
    .head(10)
    .copy()
)


# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

assert len(executive_opportunities) == 154

assert len(executive_top_10) == 10

assert (
    executive_opportunities["opportunity_rank"]
    .is_unique
)

assert (
    executive_opportunities["opportunity_rank"]
    .min() == 1
)

assert (
    executive_opportunities["opportunity_rank"]
    .max() == 154
)

assert (
    executive_opportunities["opportunity_score"]
    .is_monotonic_decreasing
)

assert (
    executive_opportunities["entity_label"]
    .notna()
    .all()
)

assert (
    executive_opportunities["recommended_action"]
    .notna()
    .all()
)


# ------------------------------------------------------------
# 6. Executive summary
# ------------------------------------------------------------

act_now_count = (
    executive_opportunities["decision"]
    .eq("ACT NOW")
    .sum()
)

prioritize_count = (
    executive_opportunities["decision"]
    .eq("PRIORITIZE")
    .sum()
)

monitor_count = (
    executive_opportunities["decision"]
    .eq("MONITOR")
    .sum()
)


print("EXECUTIVE OPPORTUNITY VIEW")
print("=" * 70)

print(
    f"Total opportunities : "
    f"{len(executive_opportunities):,}"
)

print(
    f"ACT NOW             : "
    f"{act_now_count:,}"
)

print(
    f"PRIORITIZE          : "
    f"{prioritize_count:,}"
)

print(
    f"MONITOR             : "
    f"{monitor_count:,}"
)

print(
    f"Top opportunity     : "
    f"{executive_top_10.iloc[0]['entity_label']}"
)

print(
    f"Top opportunity score: "
    f"{executive_top_10.iloc[0]['opportunity_score']:.2f}"
)


print("\nEXECUTIVE VIEW VALIDATION: PASSED")


# ------------------------------------------------------------
# 7. Display Top 10
# ------------------------------------------------------------

print("\nTOP 10 EXECUTIVE OPPORTUNITIES")
print("-" * 70)

display(
    executive_top_10
)

EXECUTIVE OPPORTUNITY VIEW
Total opportunities : 154
ACT NOW             : 7
PRIORITIZE          : 136
MONITOR             : 11
Top opportunity     : Other Visitors
Top opportunity score: 100.00

EXECUTIVE VIEW VALIDATION: PASSED

TOP 10 EXECUTIVE OPPORTUNITIES
----------------------------------------------------------------------


,opportunity_rank,opportunity_type,entity_id,entity_label,affected_visitors,opportunity_score,priority,opportunity_class,decision,decision_urgency,recommended_action
0,1,segment,Other Visitors,Other Visitors,1201156.0,100.000000,Critical,High-Reach Segment Friction,ACT NOW,Immediate,Prioritize targeted intervention for this high...
1,2,category,Uncategorized,Uncategorized,197747.0,91.191648,Critical,High-Reach Category Friction,ACT NOW,Immediate,Prioritize category-level conversion optimizat...
2,3,segment,At-Risk Visitors,At-Risk Visitors,113283.0,88.605031,Critical,High-Reach Segment Friction,ACT NOW,Immediate,Prioritize targeted intervention for this high...
3,4,segment,New Visitors,New Visitors,49219.0,84.582054,Critical,Segment Conversion Risk,ACT NOW,Immediate,Investigate segment-level conversion friction ...
4,5,category,1483,1483,40486.0,82.490397,Critical,Category Conversion Friction,ACT NOW,Immediate,Investigate category conversion friction and t...
5,6,category,491,491,37763.0,82.056702,Critical,Category Conversion Friction,ACT NOW,Immediate,Investigate category conversion friction and t...
6,7,segment,Cart Abandoners,Cart Abandoners,21323.0,80.545232,Critical,Segment Conversion Risk,ACT NOW,Immediate,Investigate segment-level conversion friction ...
7,8,category,48,48,17016.0,78.344484,High,Category Conversion Friction,PRIORITIZE,Prioritize,Investigate category conversion friction and t...
8,9,category,5,5,15036.0,77.688878,High,Category Conversion Friction,PRIORITIZE,Prioritize,Investigate category conversion friction and t...
9,10,category,1578,1578,13650.0,77.537042,High,Category Conversion Friction,PRIORITIZE,Prioritize,Investigate category conversion friction and t...


## 9. Opportunity Summary by Type

This section shows how opportunities are distributed across:

- Segment
- Product
- Category

It also summarizes affected visitors, scores, priority levels, and business decisions.

In [9]:
# ============================================================
# CELL 9 — OPPORTUNITY SUMMARY BY TYPE
# ============================================================

# Aggregate the opportunity engine output by opportunity type.
# This provides a concise business-level view of where growth
# opportunities are concentrated.


opportunity_type_summary = (
    executive_opportunities
    .groupby("opportunity_type", as_index=False)
    .agg(
        opportunity_count=("opportunity_rank", "count"),
        affected_visitors=("affected_visitors", "sum"),
        average_score=("opportunity_score", "mean"),
        maximum_score=("opportunity_score", "max"),
        critical_count=("priority", lambda x: (x == "Critical").sum()),
        high_count=("priority", lambda x: (x == "High").sum()),
        act_now_count=("decision", lambda x: (x == "ACT NOW").sum()),
        prioritize_count=("decision", lambda x: (x == "PRIORITIZE").sum()),
        monitor_count=("decision", lambda x: (x == "MONITOR").sum())
    )
    .sort_values(
        by="opportunity_count",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert (
    opportunity_type_summary["opportunity_count"].sum()
    == len(executive_opportunities)
)

assert (
    opportunity_type_summary["affected_visitors"].ge(0).all()
)

assert (
    opportunity_type_summary["average_score"].between(0, 100).all()
)

assert (
    opportunity_type_summary["maximum_score"].between(0, 100).all()
)

assert (
    opportunity_type_summary[
        "critical_count"
    ].sum()
    == (
        executive_opportunities["priority"]
        .eq("Critical")
        .sum()
    )
)

assert (
    opportunity_type_summary[
        "act_now_count"
    ].sum()
    == (
        executive_opportunities["decision"]
        .eq("ACT NOW")
        .sum()
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("OPPORTUNITY SUMMARY BY TYPE")
print("=" * 70)

print(
    f"Opportunity types : "
    f"{len(opportunity_type_summary)}"
)

print(
    f"Total opportunities: "
    f"{opportunity_type_summary['opportunity_count'].sum():,}"
)

print("\nOPPORTUNITY TYPE SUMMARY")
print("-" * 70)

display(
    opportunity_type_summary
)


print("\nTYPE SUMMARY VALIDATION: PASSED")

OPPORTUNITY SUMMARY BY TYPE
Opportunity types : 3
Total opportunities: 154

OPPORTUNITY TYPE SUMMARY
----------------------------------------------------------------------


,opportunity_type,opportunity_count,affected_visitors,average_score,maximum_score,critical_count,high_count,act_now_count,prioritize_count,monitor_count
0,product,122,62625.0,61.768748,70.936619,0,110,0,111,11
1,category,26,499303.0,76.767001,91.191648,3,23,3,23,0
2,segment,6,1395861.0,83.602553,100.000000,4,2,4,2,0



TYPE SUMMARY VALIDATION: PASSED


## 10. Priority Distribution

Priority levels are summarized within each opportunity type.

The percentage validation ensures that each opportunity type's priority distribution sums to 100%.

In [10]:
# ============================================================
# CELL 10 — PRIORITY DISTRIBUTION BY OPPORTUNITY TYPE
# ============================================================

priority_by_type = (
    executive_opportunities
    .groupby(
        ["opportunity_type", "priority"],
        as_index=False,
        observed=True
    )
    .size()
    .rename(columns={"size": "opportunity_count"})
)

priority_by_type["opportunity_share_pct"] = (
    priority_by_type["opportunity_count"]
    / priority_by_type.groupby(
        "opportunity_type",
        observed=True
    )["opportunity_count"].transform("sum")
    * 100
)

priority_by_type = (
    priority_by_type
    .sort_values(
        ["opportunity_type", "opportunity_count"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert (
    priority_by_type["opportunity_count"].sum()
    == len(executive_opportunities)
)

assert (
    priority_by_type["opportunity_share_pct"]
    .between(0, 100)
    .all()
)

type_totals = (
    priority_by_type
    .groupby(
        "opportunity_type",
        observed=True
    )["opportunity_share_pct"]
    .sum()
)

assert (
    type_totals.sub(100).abs().lt(1e-6).all()
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("PRIORITY DISTRIBUTION BY OPPORTUNITY TYPE")
print("=" * 70)

print(
    f"Total opportunities: "
    f"{len(executive_opportunities):,}"
)

print("\nPRIORITY BREAKDOWN")
print("-" * 70)

display(priority_by_type)

print("\nPRIORITY DISTRIBUTION VALIDATION: PASSED")

PRIORITY DISTRIBUTION BY OPPORTUNITY TYPE
Total opportunities: 154

PRIORITY BREAKDOWN
----------------------------------------------------------------------


,opportunity_type,priority,opportunity_count,opportunity_share_pct
0,category,High,23,88.461538
1,category,Critical,3,11.538462
2,product,High,110,90.163934
3,product,Medium,12,9.836066
4,segment,Critical,4,66.666667
5,segment,High,2,33.333333



PRIORITY DISTRIBUTION VALIDATION: PASSED


## 11. Decision Distribution

This section summarizes the business action assigned to each opportunity:

- **ACT NOW**
- **PRIORITIZE**
- **MONITOR**

In [11]:
# ============================================================
# CELL 11 — DECISION DISTRIBUTION
# ============================================================

decision_summary = (
    executive_opportunities
    .groupby("decision", as_index=False, observed=True)
    .agg(
        opportunity_count=("opportunity_rank", "count"),
        affected_visitors=("affected_visitors", "sum"),
        average_score=("opportunity_score", "mean"),
        maximum_score=("opportunity_score", "max")
    )
)

decision_summary["opportunity_share_pct"] = (
    decision_summary["opportunity_count"]
    / len(executive_opportunities)
    * 100
)

decision_order = {
    "ACT NOW": 1,
    "PRIORITIZE": 2,
    "MONITOR": 3
}

decision_summary["decision_order"] = (
    decision_summary["decision"]
    .map(decision_order)
)

decision_summary = (
    decision_summary
    .sort_values("decision_order")
    .drop(columns="decision_order")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert (
    decision_summary["opportunity_count"].sum()
    == len(executive_opportunities)
)

assert (
    decision_summary["opportunity_share_pct"]
    .between(0, 100)
    .all()
)

assert (
    abs(
        decision_summary["opportunity_share_pct"].sum()
        - 100
    ) < 1e-6
)

assert set(decision_summary["decision"]).issubset(
    {"ACT NOW", "PRIORITIZE", "MONITOR"}
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("DECISION DISTRIBUTION")
print("=" * 70)

print(
    f"Total opportunities: "
    f"{len(executive_opportunities):,}"
)

print("\nDECISION SUMMARY")
print("-" * 70)

display(decision_summary)

print("\nDECISION DISTRIBUTION VALIDATION: PASSED")

DECISION DISTRIBUTION
Total opportunities: 154

DECISION SUMMARY
----------------------------------------------------------------------


,decision,opportunity_count,affected_visitors,average_score,maximum_score,opportunity_share_pct
0,ACT NOW,7,1660977.0,87.067295,100.000000,4.545455
1,PRIORITIZE,136,293282.0,64.486833,78.344484,88.311688
2,MONITOR,11,3530.0,59.424018,59.669457,7.142857



DECISION DISTRIBUTION VALIDATION: PASSED


## 12. Top Actionable Opportunities

The top opportunities are reordered by business action so that the most urgent decisions appear first.

The final table is limited to a maximum of 10 opportunities for executive review.

In [12]:
# ============================================================
# CELL 12 — TOP ACTIONABLE OPPORTUNITIES
# ============================================================

# Rank business actions:
# ACT NOW first, then PRIORITIZE, then MONITOR.

action_order = {
    "ACT NOW": 1,
    "PRIORITIZE": 2,
    "MONITOR": 3
}

top_actionable_opportunities = (
    executive_top_10
    .assign(
        decision_order=executive_top_10["decision"].map(
            action_order
        )
    )
    .sort_values(
        ["decision_order", "opportunity_score"],
        ascending=[True, False]
    )
    .drop(columns="decision_order")
    .reset_index(drop=True)
)

# Keep maximum 10 opportunities
top_actionable_opportunities = (
    top_actionable_opportunities
    .head(10)
    .copy()
)

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

assert len(top_actionable_opportunities) <= 10

assert (
    top_actionable_opportunities["opportunity_score"]
    .between(0, 100)
    .all()
)

assert (
    top_actionable_opportunities["decision"]
    .isin(["ACT NOW", "PRIORITIZE", "MONITOR"])
    .all()
)

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("TOP ACTIONABLE OPPORTUNITIES")
print("=" * 70)

print(
    f"Opportunities displayed: "
    f"{len(top_actionable_opportunities)}"
)

print("\nTOP 10 ACTIONABLE OPPORTUNITIES")
print("-" * 70)

display(
    top_actionable_opportunities[
        [
            "opportunity_rank",
            "opportunity_type",
            "entity_id",
            "entity_label",
            "affected_visitors",
            "opportunity_score",
            "priority",
            "opportunity_class",
            "decision",
            "decision_urgency",
            "recommended_action"
        ]
    ]
)

print("\nACTIONABLE OPPORTUNITIES VALIDATION: PASSED")

TOP ACTIONABLE OPPORTUNITIES
Opportunities displayed: 10

TOP 10 ACTIONABLE OPPORTUNITIES
----------------------------------------------------------------------


,opportunity_rank,opportunity_type,entity_id,entity_label,affected_visitors,opportunity_score,priority,opportunity_class,decision,decision_urgency,recommended_action
0,1,segment,Other Visitors,Other Visitors,1201156.0,100.000000,Critical,High-Reach Segment Friction,ACT NOW,Immediate,Prioritize targeted intervention for this high...
1,2,category,Uncategorized,Uncategorized,197747.0,91.191648,Critical,High-Reach Category Friction,ACT NOW,Immediate,Prioritize category-level conversion optimizat...
2,3,segment,At-Risk Visitors,At-Risk Visitors,113283.0,88.605031,Critical,High-Reach Segment Friction,ACT NOW,Immediate,Prioritize targeted intervention for this high...
3,4,segment,New Visitors,New Visitors,49219.0,84.582054,Critical,Segment Conversion Risk,ACT NOW,Immediate,Investigate segment-level conversion friction ...
4,5,category,1483,1483,40486.0,82.490397,Critical,Category Conversion Friction,ACT NOW,Immediate,Investigate category conversion friction and t...
5,6,category,491,491,37763.0,82.056702,Critical,Category Conversion Friction,ACT NOW,Immediate,Investigate category conversion friction and t...
6,7,segment,Cart Abandoners,Cart Abandoners,21323.0,80.545232,Critical,Segment Conversion Risk,ACT NOW,Immediate,Investigate segment-level conversion friction ...
7,8,category,48,48,17016.0,78.344484,High,Category Conversion Friction,PRIORITIZE,Prioritize,Investigate category conversion friction and t...
8,9,category,5,5,15036.0,77.688878,High,Category Conversion Friction,PRIORITIZE,Prioritize,Investigate category conversion friction and t...
9,10,category,1578,1578,13650.0,77.537042,High,Category Conversion Friction,PRIORITIZE,Prioritize,Investigate category conversion friction and t...



ACTIONABLE OPPORTUNITIES VALIDATION: PASSED


## 13. Executive Action Summary

A compact management summary highlights the number of opportunities requiring immediate action, prioritization, or monitoring, along with the highest-value opportunity.

In [13]:
# ================================================================
# CELL 13 - EXECUTIVE ACTION SUMMARY
# ================================================================

print("\nEXECUTIVE ACTION SUMMARY")
print("=" * 70)

# Overall opportunity count
total_opportunities = len(top_actionable_opportunities)

# Decision distribution
decision_counts = top_actionable_opportunities["decision"].value_counts()

print(f"Actionable opportunities displayed : {total_opportunities}")

print("\nDECISION BREAKDOWN")
print("-" * 70)

for decision in ["ACT NOW", "PRIORITIZE", "MONITOR"]:
    count = decision_counts.get(decision, 0)
    print(f"{decision:<12}: {count}")

# Highest-ranked opportunity
top_opportunity = top_actionable_opportunities.iloc[0]

print("\nTOP BUSINESS OPPORTUNITY")
print("-" * 70)

print(f"Opportunity type : {top_opportunity['opportunity_type']}")
print(f"Entity          : {top_opportunity['entity_label']}")
print(f"Affected users  : {top_opportunity['affected_visitors']:,.0f}")
print(f"Opportunity score: {top_opportunity['opportunity_score']:.2f}")
print(f"Priority        : {top_opportunity['priority']}")
print(f"Decision        : {top_opportunity['decision']}")
print(f"Urgency         : {top_opportunity['decision_urgency']}")

print("\nEXECUTIVE SUMMARY VALIDATION: PASSED")


EXECUTIVE ACTION SUMMARY
Actionable opportunities displayed : 10

DECISION BREAKDOWN
----------------------------------------------------------------------
ACT NOW     : 7
PRIORITIZE  : 3
MONITOR     : 0

TOP BUSINESS OPPORTUNITY
----------------------------------------------------------------------
Opportunity type : segment
Entity          : Other Visitors
Affected users  : 1,201,156
Opportunity score: 100.00
Priority        : Critical
Decision        : ACT NOW
Urgency         : Immediate

EXECUTIVE SUMMARY VALIDATION: PASSED


## 14. Final Engine Validation

Before export, the notebook validates:

- output row count
- decision categories
- score range
- opportunity ranking

The notebook should only be considered complete when this validation passes.

In [14]:
# ================================================================
# CELL 14 - FINAL ENGINE VALIDATION
# ================================================================

print("\nFINAL GROWTH OPPORTUNITY ENGINE VALIDATION")
print("=" * 70)

# Validate actionable opportunity output
assert len(top_actionable_opportunities) == 10, \
    "Top actionable opportunity count mismatch"

# Validate decision categories
assert top_actionable_opportunities["decision"].isin(
    ["ACT NOW", "PRIORITIZE", "MONITOR"]
).all(), \
    "Invalid decision category detected"

# Validate opportunity scores
assert top_actionable_opportunities["opportunity_score"].between(
    0, 100
).all(), \
    "Invalid opportunity score detected"

# Validate ranking
assert top_actionable_opportunities["opportunity_rank"].is_monotonic_increasing, \
    "Opportunity ranking is not ordered correctly"

print("Actionable opportunities validation  : PASSED")
print("Decision category validation         : PASSED")
print("Opportunity score validation         : PASSED")
print("Opportunity ranking validation       : PASSED")

print("\nFINAL ENGINE VALIDATION: PASSED")


FINAL GROWTH OPPORTUNITY ENGINE VALIDATION
Actionable opportunities validation  : PASSED
Decision category validation         : PASSED
Opportunity score validation         : PASSED
Opportunity ranking validation       : PASSED

FINAL ENGINE VALIDATION: PASSED


## 15. Final Actionable Output Export

The validated top-10 opportunity table is exported to the project's `outputs/` directory.

This CSV is the primary machine-readable output of Notebook 08.

In [15]:
# ================================================================
# CELL 15 - FINAL ACTIONABLE OUTPUT EXPORT
# ================================================================

print("\nFINAL ACTIONABLE OUTPUT EXPORT")
print("=" * 70)

# Create a clean final output table
final_actionable_output = top_actionable_opportunities.copy()

# Export the validated actionable opportunities
final_output_path = OUTPUTS_PATH / "growth_opportunity_engine_output.csv"

final_actionable_output.to_csv(
    final_output_path,
    index=False
)

print("Final output rows      :", len(final_actionable_output))
print("Output file            :", final_output_path)
print("Export validation      : PASSED")

print("\nFINAL OUTPUT PREVIEW")
print("-" * 70)

display(final_actionable_output)


FINAL ACTIONABLE OUTPUT EXPORT
Final output rows      : 10
Output file            : D:\Data science portfolio\03_Product_Growth_Intelligence\outputs\growth_opportunity_engine_output.csv
Export validation      : PASSED

FINAL OUTPUT PREVIEW
----------------------------------------------------------------------


,opportunity_rank,opportunity_type,entity_id,entity_label,affected_visitors,opportunity_score,priority,opportunity_class,decision,decision_urgency,recommended_action
0,1,segment,Other Visitors,Other Visitors,1201156.0,100.000000,Critical,High-Reach Segment Friction,ACT NOW,Immediate,Prioritize targeted intervention for this high...
1,2,category,Uncategorized,Uncategorized,197747.0,91.191648,Critical,High-Reach Category Friction,ACT NOW,Immediate,Prioritize category-level conversion optimizat...
2,3,segment,At-Risk Visitors,At-Risk Visitors,113283.0,88.605031,Critical,High-Reach Segment Friction,ACT NOW,Immediate,Prioritize targeted intervention for this high...
3,4,segment,New Visitors,New Visitors,49219.0,84.582054,Critical,Segment Conversion Risk,ACT NOW,Immediate,Investigate segment-level conversion friction ...
4,5,category,1483,1483,40486.0,82.490397,Critical,Category Conversion Friction,ACT NOW,Immediate,Investigate category conversion friction and t...
5,6,category,491,491,37763.0,82.056702,Critical,Category Conversion Friction,ACT NOW,Immediate,Investigate category conversion friction and t...
6,7,segment,Cart Abandoners,Cart Abandoners,21323.0,80.545232,Critical,Segment Conversion Risk,ACT NOW,Immediate,Investigate segment-level conversion friction ...
7,8,category,48,48,17016.0,78.344484,High,Category Conversion Friction,PRIORITIZE,Prioritize,Investigate category conversion friction and t...
8,9,category,5,5,15036.0,77.688878,High,Category Conversion Friction,PRIORITIZE,Prioritize,Investigate category conversion friction and t...
9,10,category,1578,1578,13650.0,77.537042,High,Category Conversion Friction,PRIORITIZE,Prioritize,Investigate category conversion friction and t...


## 16. Engine Performance Summary

The final output is summarized using actionable opportunity counts, priority counts, and opportunity-score statistics.

In [16]:
# ================================================================
# CELL 16 - ENGINE PERFORMANCE SUMMARY
# ================================================================

print("\nGROWTH OPPORTUNITY ENGINE PERFORMANCE SUMMARY")
print("=" * 70)

total_opportunities = len(final_actionable_output)

act_now_count = (
    final_actionable_output["decision"] == "ACT NOW"
).sum()

prioritize_count = (
    final_actionable_output["decision"] == "PRIORITIZE"
).sum()

monitor_count = (
    final_actionable_output["decision"] == "MONITOR"
).sum()

critical_count = (
    final_actionable_output["priority"] == "Critical"
).sum()

high_count = (
    final_actionable_output["priority"] == "High"
).sum()

average_score = final_actionable_output["opportunity_score"].mean()
maximum_score = final_actionable_output["opportunity_score"].max()

print(f"Actionable opportunities analyzed : {total_opportunities}")
print(f"ACT NOW opportunities             : {act_now_count}")
print(f"PRIORITIZE opportunities          : {prioritize_count}")
print(f"MONITOR opportunities             : {monitor_count}")
print(f"Critical opportunities            : {critical_count}")
print(f"High-priority opportunities       : {high_count}")
print(f"Average opportunity score        : {average_score:.2f}")
print(f"Maximum opportunity score        : {maximum_score:.2f}")

print("\nENGINE PERFORMANCE VALIDATION: PASSED")


GROWTH OPPORTUNITY ENGINE PERFORMANCE SUMMARY
Actionable opportunities analyzed : 10
ACT NOW opportunities             : 7
PRIORITIZE opportunities          : 3
MONITOR opportunities             : 0
Critical opportunities            : 7
High-priority opportunities       : 3
Average opportunity score        : 84.30
Maximum opportunity score        : 100.00

ENGINE PERFORMANCE VALIDATION: PASSED


## 17. Final Business Insights

The exported final output is reloaded and independently validated before producing the final business-facing interpretation.

This provides a clean separation between **engine output** and **business insight generation**.

In [17]:
# ================================================================
# CELL 17 - FINAL BUSINESS INSIGHTS
# ================================================================

import pandas as pd

print("\nFINAL BUSINESS INSIGHTS")
print("=" * 70)

# Load the validated output created in Cell 15
business_output = pd.read_csv(
    OUTPUTS_PATH / "growth_opportunity_engine_output.csv"
)

# Basic validation
required_columns = [
    "opportunity_type",
    "entity_label",
    "affected_visitors",
    "opportunity_score",
    "priority",
    "decision",
    "decision_urgency"
]

assert all(
    column in business_output.columns
    for column in required_columns
), "Required output columns are missing"

assert len(business_output) == 10, \
    "Expected 10 actionable opportunities"

# Summary values
total_actionable = len(business_output)

act_now_count = (
    business_output["decision"] == "ACT NOW"
).sum()

prioritize_count = (
    business_output["decision"] == "PRIORITIZE"
).sum()

monitor_count = (
    business_output["decision"] == "MONITOR"
).sum()

average_score = business_output["opportunity_score"].mean()
maximum_score = business_output["opportunity_score"].max()

# Top opportunity
top_opportunity = business_output.iloc[0]

print("\nTOP OPPORTUNITY")
print("-" * 70)

print("Opportunity type :", top_opportunity["opportunity_type"])
print("Entity           :", top_opportunity["entity_label"])
print(
    "Affected users   :",
    f'{top_opportunity["affected_visitors"]:,.0f}'
)
print(
    "Opportunity score:",
    f'{top_opportunity["opportunity_score"]:.2f}'
)
print("Priority         :", top_opportunity["priority"])
print("Decision         :", top_opportunity["decision"])
print("Urgency          :", top_opportunity["decision_urgency"])

print("\nDECISION INSIGHT")
print("-" * 70)

print(
    f"{act_now_count} of the top {total_actionable} actionable "
    "opportunities require immediate action."
)

print(
    f"{prioritize_count} opportunities should be prioritized "
    "for targeted optimization."
)

print(
    f"{monitor_count} opportunities should be monitored."
)

print("\nSCORE INSIGHT")
print("-" * 70)

print(
    f"Average opportunity score : {average_score:.2f}"
)

print(
    f"Highest opportunity score : {maximum_score:.2f}"
)

print("\nBUSINESS INSIGHTS VALIDATION: PASSED")


FINAL BUSINESS INSIGHTS

TOP OPPORTUNITY
----------------------------------------------------------------------
Opportunity type : segment
Entity           : Other Visitors
Affected users   : 1,201,156
Opportunity score: 100.00
Priority         : Critical
Decision         : ACT NOW
Urgency          : Immediate

DECISION INSIGHT
----------------------------------------------------------------------
7 of the top 10 actionable opportunities require immediate action.
3 opportunities should be prioritized for targeted optimization.
0 opportunities should be monitored.

SCORE INSIGHT
----------------------------------------------------------------------
Average opportunity score : 84.30
Highest opportunity score : 100.00

BUSINESS INSIGHTS VALIDATION: PASSED


## Final Notebook Status

### Validation checklist

- [x] Upstream input contracts validated
- [x] Opportunity signals normalized
- [x] Unified opportunity table created
- [x] Opportunity scores validated
- [x] Opportunity classifications validated
- [x] Executive opportunity view validated
- [x] Top actionable opportunities validated
- [x] Final engine validation passed
- [x] Final CSV export generated
- [x] Business insights validated

### Final deliverable

**`outputs/growth_opportunity_engine_output.csv`**

This notebook is designed as the final analytical decision-support layer before downstream dashboard/application integration.
